In [2]:
from datasets import load_dataset
import pandas as pd

# Download the banking77 dataset
dataset = load_dataset("banking77")

# Convert train and test sets to Pandas DataFrames
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])

# Print dataset statistics
print(f"Training samples: {len(df_train)}")
print(f"Testing samples: {len(df_test)}")

# Display the first 5 rows for inspection
display(df_train.head())

/Users/trungmin/roughset-intent-classification/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training samples: 10003
Testing samples: 3080


,text,label
0,I am still waiting on my card?,11
1,What can I do if my card still hasn't arrived ...,11
2,I have been waiting over a week. Is the card s...,11
3,Can I track my card while it is in the process...,11
4,"How do I know if I will get my card, or if it ...",11


In [2]:
# Extract intent names from dataset metadata
label_names = dataset['train'].features['label'].names

# Create a mapping dictionary (e.g., 11 -> 'card_arrival')
id2label = {i: name for i, name in enumerate(label_names)}

# Map IDs to human-readable intent names in both DataFrames
df_train['intent_name'] = df_train['label'].map(id2label)
df_test['intent_name'] = df_test['label'].map(id2label)

# Display results for verification
display(df_train[['text', 'label', 'intent_name']].head())

,text,label,intent_name
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


In [ ]:
import sys
import numpy as np
import os

# Add src directory to path for module resolution
sys.path.append(os.path.abspath('./src'))
from embeddings import TextEmbedder

# Initialize the embedder
embedder = TextEmbedder()

# Test on the first 1000 sentences of the training set (to verify pipeline speed)
sample_texts = df_train['text'].tolist()[:1000]
X_vectors = embedder.encode_texts(sample_texts)

# Inspect the resulting matrix structure
print("Matrix shape for the first 1000 sentences:", X_vectors.shape)
print("Vector for the first sentence (first 5 dimensions):", X_vectors[0][:5])

Loading Sentence-Transformers model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9436.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 1000 sentences into vectors...


Batches: 100%|██████████| 32/32 [00:03<00:00,  9.51it/s]

Success! Vector matrix shape: (1000, 384)
Matrix shape for the first 1000 sentences: (1000, 384)
Vector for the first sentence (first 5 dimensions): [-0.03535237 -0.0421367  -0.00275709  0.03193695  0.02319006]


In [5]:
import sys
import os
import pandas as pd
from datasets import load_dataset

# 1. PATH CONFIGURATION
# Adding 'src' to system path for module resolution
sys.path.append(os.path.abspath('./src'))
sys.path.append(os.path.abspath('../src'))

from embeddings import TextEmbedder
from rough_set import RoughSetClassifier


# PART 0: DATA PREPARATION & LABEL MAPPING

print("--- STEP 0: DATA LOADING & LABEL MAPPING ---")
dataset = load_dataset("banking77")
df_train = pd.DataFrame(dataset['train'])

# Initialize intent names and mapping dictionary to prevent NameError
label_names = dataset['train'].features['label'].names
id2label = {i: name for i, name in enumerate(label_names)}
print("Intent dictionary loaded successfully.")


# PART 1: VECTOR EXTRACTION

print("\n--- STEP 1: VECTOR EXTRACTION ---")
embedder = TextEmbedder()

# Select the first 1000 samples for encoding
sample_texts = df_train['text'].tolist()[:1000]
X_vectors = embedder.encode_texts(sample_texts)


# PART 2: ROUGH SET CORE PROCESSING

print("\n--- STEP 2: ROUGH SET CLASSIFICATION ---")
sample_labels = df_train['label'].tolist()[:1000]

# Initialize and fit the model
# Partitioning data into 50 granules with an 85% purity threshold
rs_model = RoughSetClassifier(n_clusters=50, lower_threshold=0.85)
rs_model.fit(X_vectors, sample_labels)

# Test predictions on the first 5 samples
test_vectors = X_vectors[:5]
predictions = rs_model.predict(test_vectors)

print("\n--- PREDICTION RESULTS ---")
for i, (pred_labels, region_type) in enumerate(predictions):
    text = sample_texts[i]
    # Map numerical IDs back to human-readable intent strings
    suggested_intents = [id2label[lbl] for lbl in pred_labels]
    
    print(f"Query: {text}")
    print(f"Region Classification: {region_type}")
    print(f"Suggested Intents: {suggested_intents}\n")

--- STEP 0: DATA LOADING & LABEL MAPPING ---
Intent dictionary loaded successfully.

--- STEP 1: VECTOR EXTRACTION ---
Loading Sentence-Transformers model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7125.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 1000 sentences into vectors...


Batches: 100%|██████████| 32/32 [00:00<00:00, 78.04it/s]


Success! Vector matrix shape: (1000, 384)

--- STEP 2: ROUGH SET CLASSIFICATION ---
Generating 50 knowledge granules using K-Means...
Successfully constructed Lower Approximations and Boundary Regions.

--- PREDICTION RESULTS ---
Query: I am still waiting on my card?
Region Classification: CERTAIN
Suggested Intents: ['card_arrival']

Query: What can I do if my card still hasn't arrived after 2 weeks?
Region Classification: CERTAIN
Suggested Intents: ['card_arrival']

Query: I have been waiting over a week. Is the card still coming?
Region Classification: CERTAIN
Suggested Intents: ['card_arrival']

Query: Can I track my card while it is in the process of delivery?
Region Classification: CERTAIN
Suggested Intents: ['card_arrival']

Query: How do I know if I will get my card, or if it is lost?
Region Classification: CERTAIN
Suggested Intents: ['card_linking']



In [6]:
import sys
import os

# 1. PATH CONFIGURATION
# Adding 'src' to system path for module resolution
sys.path.append(os.path.abspath('./src'))
sys.path.append(os.path.abspath('../src'))

from evaluation import RoughSetEvaluator

print("--- LARGE-SCALE EXPERIMENT ---")

# 1. DATA PREPARATION
# Training set: 5000 samples | Test set: 1000 samples
train_texts = df_train['text'].tolist()[:5000]
train_labels = df_train['label'].tolist()[:5000]

test_texts = df_test['text'].tolist()[:1000]
test_labels = df_test['label'].tolist()[:1000]

# 2. FEATURE EXTRACTION (VECTOR EMBEDDING)
print("\nEncoding training set vectors...")
X_train = embedder.encode_texts(train_texts)

print("Encoding test set vectors...")
X_test = embedder.encode_texts(test_texts)

# 3. MODEL TRAINING
# Increasing granularity to 300 clusters for 5000 samples to achieve finer knowledge granules
print("\nTraining Rough Set Model...")
rs_model_large = RoughSetClassifier(n_clusters=300, lower_threshold=0.85)
rs_model_large.fit(X_train, train_labels)

# 4. INFERENCE ON TEST SET
print("Generating predictions for test set...")
test_predictions = rs_model_large.predict(X_test)

# 5. PERFORMANCE EVALUATION
evaluator = RoughSetEvaluator()
metrics = evaluator.evaluate(test_predictions, test_labels)

--- LARGE-SCALE EXPERIMENT ---

Encoding training set vectors...
Encoding 5000 sentences into vectors...


Batches: 100%|██████████| 157/157 [00:03<00:00, 40.22it/s]


Success! Vector matrix shape: (5000, 384)
Encoding test set vectors...
Encoding 1000 sentences into vectors...


Batches: 100%|██████████| 32/32 [00:00<00:00, 51.39it/s]


Success! Vector matrix shape: (1000, 384)

Training Rough Set Model...
Generating 300 knowledge granules using K-Means...
Successfully constructed Lower Approximations and Boundary Regions.
Generating predictions for test set...

ROUGH SET MODEL EVALUATION REPORT
Total Test Samples        : 1000
Certain Region Samples    : 830 (83.00%)
Boundary Region Samples   : 170 (17.00%)
--------------------------------------------------
Certainty Accuracy (Target): 97.35%
